# Substantia nigra caQTLs

In [ ]:
import pandas as pd
import polars as pl
import pickle
import os
import re
import glob
import sys
import matplotlib.pyplot as pltplo
from pycisTopic.cistopic_class import *
from pycisTopic.clust_vis import *
import pickle
import gzip

In [ ]:
#set up
proj_dir = 'snATAC_analysis/'
proj_subdir = proj_dir + '/SN/caQTL'
out = proj_subdir + 'out/'
tmp = '/scratch/'
data_dir = 'data/'

## 1. Prepare count matrices

In [ ]:
infile = open(out + 'merged_SN_T2T_CT2.pkl', 'rb')
cistopic_obj = pickle.load(infile)
infile.close()

In [ ]:
## add a column combining donor id and cell type
cistopic_obj.cell_data['donor_ct'] = cistopic_obj.cell_data['donor_id'] + '__' + cistopic_obj.cell_data['CISTOPIC_2_final_annot_NA']
## also combine neuron subtypes into one neuron class
cistopic_obj.cell_data['CISTOPIC_2_final_annot_neurons'] = cistopic_obj.cell_data['CISTOPIC_2_final_annot_NA'].apply(lambda x: 'neuron' if x in ['GabaN','DopaN','GlutaN'] else x)
cistopic_obj.cell_data['donor_ct_neurons'] = cistopic_obj.cell_data['donor_id'] + '__' + cistopic_obj.cell_data['CISTOPIC_2_final_annot_neurons']

### 1.1 per cell type

In [ ]:
# filter cell data to keep only cells from the SN cell types and cells that are assigned to donors
filtered_data = cistopic_obj.cell_data[
    cistopic_obj.cell_data['donor_ct'].notna() & 
    ~cistopic_obj.cell_data['donor_ct'].fillna('').str.contains('doublet') & 
    ~cistopic_obj.cell_data['donor_ct'].fillna('').str.contains('L2') &
    ~cistopic_obj.cell_data['donor_ct'].fillna('').str.contains('unassigned')
]

In [19]:
filtered_data.shape

(1225921, 41)

In [ ]:
## Add additional filter where donor cell type count should be above 50
low_freq_donors = filtered_data.donor_ct.value_counts()
low_freq_donors = low_freq_donors[low_freq_donors < 50].index.tolist()
filtered_data = filtered_data[~filtered_data.donor_ct.isin(low_freq_donors)]
filtered_data.shape

In [ ]:
# get the barcodes of cells per donor_ct
donor_cell_type_bcs = {
    donor_ct: group.index.tolist()
    for donor_ct, group in filtered_data.groupby('donor_ct')
}

In [ ]:
# get the indices of cells per donor_ct: from fragment matrix

donor_cell_type_indices = {
    donor_ct: [
        cistopic_obj.cell_data.index.get_loc(bc)
        for bc in barcodes
    ]
    for donor_ct, barcodes in donor_cell_type_bcs.items()
}

In [23]:
# check if the length of the number of barcodes matches the indices lenghts for each donor_ct combination
for key in donor_cell_type_bcs.keys():
    print(f'{key}: {len(donor_cell_type_bcs[key])} -- {len(donor_cell_type_indices[key])}')

ASA_001__Astro: 1251 -- 1251
ASA_001__DopaN: 288 -- 288
ASA_001__Endo: 170 -- 170
ASA_001__GabaN: 515 -- 515
ASA_001__Micro-PVM: 1820 -- 1820
ASA_001__OPC: 369 -- 369
ASA_001__Oligo: 1707 -- 1707
ASA_001__immune: 53 -- 53
ASA_002__Astro: 5065 -- 5065
ASA_002__DopaN: 106 -- 106
ASA_002__Endo: 57 -- 57
ASA_002__GabaN: 494 -- 494
ASA_002__Micro-PVM: 3339 -- 3339
ASA_002__OPC: 1835 -- 1835
ASA_002__Oligo: 7437 -- 7437
ASA_002__immune: 65 -- 65
ASA_003__Astro: 4357 -- 4357
ASA_003__DopaN: 73 -- 73
ASA_003__Endo: 63 -- 63
ASA_003__GabaN: 240 -- 240
ASA_003__Micro-PVM: 2058 -- 2058
ASA_003__OPC: 1345 -- 1345
ASA_003__Oligo: 7500 -- 7500
ASA_004__Astro: 7129 -- 7129
ASA_004__DopaN: 181 -- 181
ASA_004__Endo: 94 -- 94
ASA_004__GabaN: 1605 -- 1605
ASA_004__GlutaN: 316 -- 316
ASA_004__Micro-PVM: 2900 -- 2900
ASA_004__OPC: 1648 -- 1648
ASA_004__Oligo: 9860 -- 9860
ASA_004__immune: 81 -- 81
ASA_005__Astro: 7623 -- 7623
ASA_005__DopaN: 387 -- 387
ASA_005__Endo: 66 -- 66
ASA_005__GabaN: 265 -- 265
ASA

In [ ]:
# get the indices per cell type
cts_indices = {}
for k, indices in donor_cell_type_indices.items():
    ct = k.split('__')[1]
    cts_indices.setdefault(ct, []).extend(indices)

In [25]:
# cells per cell type
for k in cts_indices.keys():
    print(f'{k}: {len(cts_indices[k])}')

Astro: 309673
DopaN: 5634
Endo: 3563
GabaN: 50253
Micro-PVM: 168674
OPC: 72135
Oligo: 600645
immune: 2381
GlutaN: 6382


In [26]:
# get a dict with the region coordinates as keys and an identifier region_x that indicates the index of the region in region_names
region_dict = {}
for i in range(len(cistopic_obj.region_names)):
    region_dict[cistopic_obj.region_names[i]] = i

In [28]:
save_dir = proj_subdir + 'data/'
with open(save_dir + 'regions_dict.pkl', 'wb') as file:
    pickle.dump(region_dict, file)

In [ ]:
# split fragment matrix by cell types to keep only cells from the same cell type
fragment_matrices = {}
for ct, indices in cts_indices.items():
    print(ct)
    fragment_matrices[ct] = cistopic_obj.fragment_matrix[:, indices]

In [ ]:
## get indices of cells per donor and cell type
donor_subindex = {}
for celltype, cpm_matrix in fragment_matrices.items():
    donor_subindex[celltype] = {
        'ind_matches': {ind: i for i, ind in enumerate(cts_indices[celltype])},
        'donor': {}
    }
    
    # Precompute the indices for the current cell type
    ind_matches = donor_subindex[celltype]['ind_matches']
    
    for k, indices in donor_cell_type_indices.items():
        donor, ct = k.split('__')
        if ct == celltype:
            # Filter and map indices efficiently
            donor_subindex[celltype]['donor'][donor] = [
                ind_matches[donor_ind] for donor_ind in indices if donor_ind in ind_matches
            ]

In [ ]:
##sanity check 
for celltype in donor_subindex.keys():
    print(celltype)
    for k, indices in donor_cell_type_indices.items():
        donor, ct = k.split('__')
        if ct == celltype:
            ind_len = len(indices)
            dict_len = len(donor_subindex[celltype]['donor'][donor])
            if ind_len != dict_len:
                print(f'MISTAKE {donor}: len in donor_cell_type_indices = {len(indices)} - {dict_len} = len in immune_test_dict')

In [ ]:
import pickle
with open(save_dir + 'donor_subindex.pkl', 'wb') as file:
    pickle.dump(donor_subindex, file)

#### process matrix
- one matrix per cell type
- regions in the rows, donors in the columns
- values are sums of the fragment counts of all cells of one cell type, for one donor and one region
- counts are cpm normalized and logtransformed

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

In [ ]:
# Create the dictionary to store the matrices
ct_region_donor_per_ct_matrices = {}

for ct, donor_data in donor_subindex.items():
    print(ct)
    
    ct_region_donor_per_ct_matrices[ct] = {}
    
    # Access donor indices directly
    donor_cell_indices = donor_data['donor']
    
    # Access the corresponding fragment matrix
    data = fragment_matrices[ct]
    regions = data.shape[0]
    
    # Filter donors where the number of indices is >= 50
    filtered_donors = {donor: indices for donor, indices in donor_cell_indices.items() if len(indices) >= 50}
    donors = list(filtered_donors.keys())  # List of filtered donor names
    
    print(donors)
    print(len(donors))
    
    ct_region_donor_per_ct_matrices[ct]['donors_filtered'] = donors
    
    # Pre-allocate result array for summed expression
    result = np.zeros((regions, len(donors)))  # Matrix to store the summed expression per donor
    
    # Check if data is a sparse matrix
    if isinstance(data, csr_matrix):
        # Efficient sparse matrix summing
        for i, donor in enumerate(donors):
            print(donor)
            indices = filtered_donors[donor]  # Get the cell indices for the current donor
            
            # Use sparse matrix summing: this avoids converting the sparse matrix to dense
            result[:, i] = data[:, indices].sum(axis=1).A.flatten()  # .A converts to a dense array if needed
    else:
        # Fallback for non-sparse data (dense matrices)
        for i, donor in enumerate(donors):
            print(donor)
            indices = filtered_donors[donor]  # Get the cell indices for the current donor
            
            # Sum columns for dense matrix (as before)
            result[:, i] = data[:, indices].sum(axis=1).flatten()
    
    print(result.shape)
    ct_region_donor_per_ct_matrices[ct]['matrix'] = result

In [ ]:
for k in ct_region_donor_per_ct_matrices.keys():
    ndonors = len(ct_region_donor_per_ct_matrices[k]['donors_filtered'])
    shape = ct_region_donor_per_ct_matrices[k]['matrix'].shape
    print(f'{k}: ndonors = {ndonors}, shape = {shape}')

In [ ]:
## parse regions
region_keys = list(region_dict.keys())
region_data = [
    (key.split(':')[0],  # Chromosome
     key.split(':')[1].split('-')[0],  # Start
     key.split(':')[1].split('-')[1],  # End
     key)  # Region ID
    for key in region_keys
]
region_df = pd.DataFrame(region_data, columns=['#chr', 'start', 'end', 'region_id'])

In [ ]:
def cpm_normalize_and_log_transform(fragment_matrix, log1p=True):
    """
    Normalize a fragment matrix to Counts Per Million (CPM) and apply log transformation.

    Args:
        fragment_matrix (scipy.sparse.csr_matrix or numpy.ndarray): 
            Fragment matrix of shape (features, cells).
        log1p (bool): If True, apply log1p (log(1 + x)) transformation after CPM normalization.

    Returns:
        Same type as input (scipy.sparse.csr_matrix or numpy.ndarray): 
            Log-transformed CPM normalized fragment matrix.
    """
    # Compute column sums (library sizes)
    if isinstance(fragment_matrix, csr_matrix):
        column_sums = fragment_matrix.sum(axis=0).A1  # For sparse matrix
    elif isinstance(fragment_matrix, np.ndarray):
        column_sums = fragment_matrix.sum(axis=0)  # For dense array
    else:
        raise ValueError("Input must be a scipy.sparse.csr_matrix or numpy.ndarray.")

    # Avoid division by zero
    column_sums[column_sums == 0] = 1

    # CPM normalization
    if isinstance(fragment_matrix, csr_matrix):
        cpm_normalized_matrix = fragment_matrix.multiply(1e6 / column_sums)
    else:  # For dense matrix
        cpm_normalized_matrix = (fragment_matrix * 1e6) / column_sums

    # Log transformation
    if log1p:
        if isinstance(cpm_normalized_matrix, csr_matrix):
            # For sparse matrix, apply log1p element-wise
            cpm_normalized_matrix = cpm_normalized_matrix.log1p()
        else:
            # For dense matrix, use NumPy's log1p
            cpm_normalized_matrix = np.log1p(cpm_normalized_matrix)

    return cpm_normalized_matrix

def create_logtransformed_cpm_df(region_df, cpm_matrix, donor_names):
    """
    Create a new DataFrame by adding log-transformed CPM columns to the region DataFrame.

    Args:
        region_df (pd.DataFrame): Original DataFrame containing region information.
        cpm_matrix (scipy.sparse.csr_matrix or np.ndarray): Log-transformed CPM matrix.
        donor_names (list): List of donor names corresponding to the columns of the CPM matrix.

    Returns:
        pd.DataFrame: A new DataFrame with the original region columns and donor CPM columns.
    """
    # Ensure donor names match the CPM matrix columns
    if len(donor_names) != cpm_matrix.shape[1]:
        raise ValueError("Number of donor names must match the number of columns in the CPM matrix.")

    # Convert sparse matrix to dense if necessary
    if isinstance(cpm_matrix, csr_matrix):
        cpm_matrix = cpm_matrix.toarray()

    # Create a new DataFrame with the donor columns
    donor_df = pd.DataFrame(cpm_matrix, columns=donor_names)

    # Concatenate the region DataFrame with the donor DataFrame
    new_df = pd.concat([region_df.reset_index(drop=True), donor_df], axis=1)

    return new_df

def sort_by_chromosome_and_start(df):
    # First, ensure the dataframe is sorted by chromosome and start position
    df['#chr'] = df['#chr'].astype(str)  # Make sure '#chr' is treated as a string
    return df.sort_values(by=['#chr', 'start'])

In [ ]:
cpm_fragment_matrices_log = {}
for ct, data in loaded_ct_region_donor_per_ct_matrices.items():
    cpm_fragment_matrices_log[ct] = cpm_normalize_and_log_transform(data['matrix'])

In [ ]:
bed_files = {}
for ct in cpm_fragment_matrices_log.keys():
    print(ct)
    donor_names = list(ct_region_donor_per_ct_matrices[ct]['donors_filtered'])
    bed_files[ct] = create_logtransformed_cpm_df(region_df, cpm_fragment_matrices_log[ct], donor_names)
    print(cpm_fragment_matrices_log[ct].shape)
    print(bed_files[ct].shape)

In [ ]:
## format bed files
for ct in bed_files.keys():
    bed_files[ct]['start'] = pd.to_numeric(bed_files[ct]['start'], errors='coerce', downcast='integer')
    bed_files[ct]['end'] = pd.to_numeric(bed_files[ct]['end'], errors='coerce', downcast='integer')

bed_files_sorted = {}
for ct in bed_files.keys():
    bed_files_sorted[ct] = sort_by_chromosome_and_start(bed_files[ct])

In [62]:
for ct in bed_files_sorted.keys():
    output_filename = f"{ct}_counts_summed_per_donor_sorted_cpm_log1p.bed"
    bed_files_sorted[ct].to_csv(save_dir + output_filename, sep='\t', header=True, index=False)

### 1.2 combined neurons

In [ ]:
filtered_data = cistopic_obj.cell_data[
    cistopic_obj.cell_data['donor_ct_neurons'].notna() & 
    cistopic_obj.cell_data['donor_ct_neurons'].str.contains('neuron', case=False, na=False) & 
    ~cistopic_obj.cell_data['donor_ct_neurons'].str.contains('doublet|unassigned', case=False, na=False)
]

In [14]:
filtered_data.shape

(65435, 41)

In [18]:
low_freq_donors = filtered_data.donor_ct_neurons.value_counts()
low_freq_donors = low_freq_donors[low_freq_donors < 50].index.tolist()
filtered_data = filtered_data[~filtered_data.donor_ct_neurons.isin(low_freq_donors)]
filtered_data.shape

(64907, 41)

In [19]:
# get the barcodes of cells per donor_ct
donor_cell_type_bcs = {
    donor_ct: group.index.tolist()
    for donor_ct, group in filtered_data.groupby('donor_ct_neurons')
}

In [20]:
# get the indices of cells per donor_ct: from fragment matrix

donor_cell_type_indices = {
    donor_ct: [
        cistopic_obj.cell_data.index.get_loc(bc)
        for bc in barcodes
    ]
    for donor_ct, barcodes in donor_cell_type_bcs.items()
}

In [21]:
# check if the length of the number of barcodes matches the indices lenghts for each donor_ct combination
for key in donor_cell_type_bcs.keys():
    print(f'{key}: {len(donor_cell_type_bcs[key])} -- {len(donor_cell_type_indices[key])}')

ASA_001__neuron: 805 -- 805
ASA_002__neuron: 603 -- 603
ASA_003__neuron: 313 -- 313
ASA_004__neuron: 2102 -- 2102
ASA_005__neuron: 720 -- 720
ASA_006__neuron: 340 -- 340
ASA_007__neuron: 119 -- 119
ASA_008__neuron: 197 -- 197
ASA_009__neuron: 297 -- 297
ASA_010__neuron: 373 -- 373
ASA_011__neuron: 654 -- 654
ASA_012__neuron: 3987 -- 3987
ASA_013__neuron: 2536 -- 2536
ASA_014__neuron: 1235 -- 1235
ASA_015__neuron: 795 -- 795
ASA_016__neuron: 76 -- 76
ASA_017__neuron: 91 -- 91
ASA_018__neuron: 108 -- 108
ASA_019__neuron: 140 -- 140
ASA_020__neuron: 636 -- 636
ASA_021__neuron: 78 -- 78
ASA_023__neuron: 56 -- 56
ASA_024__neuron: 153 -- 153
ASA_025__neuron: 210 -- 210
ASA_026__neuron: 1290 -- 1290
ASA_027__neuron: 886 -- 886
ASA_028__neuron: 2474 -- 2474
ASA_029__neuron: 936 -- 936
ASA_030__neuron: 543 -- 543
ASA_031__neuron: 6126 -- 6126
ASA_032__neuron: 1778 -- 1778
ASA_033__neuron: 3091 -- 3091
ASA_034__neuron: 285 -- 285
ASA_035__neuron: 4821 -- 4821
ASA_036__neuron: 652 -- 652
ASA_037_

In [22]:
# get the indices per cell type
cts_indices = {}
for k, indices in donor_cell_type_indices.items():
    ct = k.split('__')[1]
    cts_indices.setdefault(ct, []).extend(indices)

In [23]:
# cells per cell type
for k in cts_indices.keys():
    print(f'{k}: {len(cts_indices[k])}')

neuron: 64907


In [ ]:
# split fragment matrix by cell types to keep only cells from the same cell type
fragment_matrices = {}
for ct, indices in cts_indices.items():
    print(ct)
    fragment_matrices[ct] = cistopic_obj.fragment_matrix[:, indices]

In [ ]:
## Do for all cell types
donor_subindex = {}
for celltype, cpm_matrix in fragment_matrices.items():
    donor_subindex[celltype] = {
        'ind_matches': {ind: i for i, ind in enumerate(cts_indices[celltype])},
        'donor': {}
    }
    
    # Precompute the indices for the current cell type
    ind_matches = donor_subindex[celltype]['ind_matches']
    
    for k, indices in donor_cell_type_indices.items():
        donor, ct = k.split('__')
        if ct == celltype:
            # Filter and map indices efficiently
            donor_subindex[celltype]['donor'][donor] = [
                ind_matches[donor_ind] for donor_ind in indices if donor_ind in ind_matches
            ]

In [ ]:
for celltype in donor_subindex.keys():
    print(celltype)
    for k, indices in donor_cell_type_indices.items():
        donor, ct = k.split('__')
        if ct == celltype:
            ind_len = len(indices)
            dict_len = len(donor_subindex[celltype]['donor'][donor])
            if ind_len != dict_len:
                print(f'MISTAKE {donor}: len in donor_cell_type_indices = {len(indices)} - {dict_len} = len in immune_test_dict')

In [ ]:
with open(save_dir + 'donor_subindex_neurons.pkl', 'wb') as file:
    pickle.dump(donor_subindex, file)

#### process matrix

In [ ]:
# Create the dictionary to store the matrices
ct_region_donor_per_ct_matrices = {}

for ct, donor_data in donor_subindex.items():
    print(ct)
    
    ct_region_donor_per_ct_matrices[ct] = {}
    
    # Access donor indices directly
    donor_cell_indices = donor_data['donor']
    
    # Access the corresponding fragment matrix
    data = fragment_matrices[ct]
    regions = data.shape[0]
    
    # Filter donors where the number of indices is >= 50
    filtered_donors = {donor: indices for donor, indices in donor_cell_indices.items() if len(indices) >= 50}
    donors = list(filtered_donors.keys())  # List of filtered donor names
    
    print(donors)
    print(len(donors))
    
    ct_region_donor_per_ct_matrices[ct]['donors_filtered'] = donors
    
    # Pre-allocate result array for summed expression
    result = np.zeros((regions, len(donors)))  # Matrix to store the summed expression per donor
    
    # Check if data is a sparse matrix
    if isinstance(data, csr_matrix):
        # Efficient sparse matrix summing
        for i, donor in enumerate(donors):
            print(donor)
            indices = filtered_donors[donor]  # Get the cell indices for the current donor
            
            # Use sparse matrix summing: this avoids converting the sparse matrix to dense
            result[:, i] = data[:, indices].sum(axis=1).A.flatten()  # .A converts to a dense array if needed
    else:
        # Fallback for non-sparse data (dense matrices)
        for i, donor in enumerate(donors):
            print(donor)
            indices = filtered_donors[donor]  # Get the cell indices for the current donor
            
            # Sum columns for dense matrix (as before)
            result[:, i] = data[:, indices].sum(axis=1).flatten()
    
    print(result.shape)
    ct_region_donor_per_ct_matrices[ct]['matrix'] = result

In [ ]:
for k in ct_region_donor_per_ct_matrices.keys():
    ndonors = len(ct_region_donor_per_ct_matrices[k]['donors_filtered'])
    shape = ct_region_donor_per_ct_matrices[k]['matrix'].shape
    print(f'{k}: ndonors = {ndonors}, shape = {shape}')

In [ ]:
cpm_fragment_matrices_log = {}
for ct, data in loaded_ct_region_donor_per_ct_matrices.items():
    cpm_fragment_matrices_log[ct] = cpm_normalize_and_log_transform(data['matrix'])

In [ ]:
bed_files = {}
for ct in cpm_fragment_matrices_log.keys():
    print(ct)
    donor_names = list(ct_region_donor_per_ct_matrices[ct]['donors_filtered'])
    bed_files[ct] = create_logtransformed_cpm_df(region_df, cpm_fragment_matrices_log[ct], donor_names)
    print(cpm_fragment_matrices_log[ct].shape)
    print(bed_files[ct].shape)

In [ ]:
## format bed files
for ct in bed_files.keys():
    bed_files[ct]['start'] = pd.to_numeric(bed_files[ct]['start'], errors='coerce', downcast='integer')
    bed_files[ct]['end'] = pd.to_numeric(bed_files[ct]['end'], errors='coerce', downcast='integer')

bed_files_sorted = {}
for ct in bed_files.keys():
    bed_files_sorted[ct] = sort_by_chromosome_and_start(bed_files[ct])

In [62]:
for ct in bed_files_sorted.keys():
    output_filename = f"{ct}_counts_summed_per_donor_sorted_cpm_log1p.bed"
    bed_files_sorted[ct].to_csv(save_dir + output_filename, sep='\t', header=True, index=False)

## 2. prepare covariates
- age
- sex
- donor biobank
- sequencing modality
- 5 genotype PCs
- 30 data PCs

In [ ]:
loaded_ct_region_donor_per_ct_matrices = {}
for ct in cts:
    file_path = save_dir + f'{ct}_per_donor_summed_counts5.pkl.gz'
    
    with gzip.open(file_path, 'rb') as file:
        loaded_ct_region_donor_per_ct_matrices[ct] = pickle.load(file)

print("Data successfully loaded for the following cell types:")
print(loaded_ct_region_donor_per_ct_matrices.keys())

### Add donor information

In [ ]:
donor_metadata_path = data_dir + 'sn_donor_metadata.csv'
donor_metadata = pd.read_csv(donor_metadata_path, index_col = 0,  sep = '\t')

In [ ]:
age = donor_metadata['age_at_collection'].astype('int').tolist()

## encode sex: male = 1 - female = 0
sex = donor_metadata['sex'].tolist()
sex_encoded = []
for x in sex:
    if x == 'Male':
        sex_encoded.append(1)
    else:
        sex_encoded.append(0)

##encode biobank QSBB UK = 0, Edi = 0
biobank = donor_metadata['biobank'].tolist()
biobank_q_encoded = []
for x in biobank:
    if x == 'QSBB_UK':
        biobank_q_encoded.append(1)
    else:
        biobank_q_encoded.append(0)

In [ ]:
donor_info = pd.DataFrame(columns=subjects)
donor_info.loc['age_at_collection'] = age
donor_info.loc['sex_Male'] = sex_encoded
donor_info.loc['biobank_name_QSBB_UK'] = biobank_q_encoded

In [ ]:
## add technology as covariable: MO, scATAC, scaleATAC 
metadata_path = data_dir + 'atac_samples_metadata_with_donor_region_info.csv"
metadata = pl.read_csv(metadata_path)
metadata.filter(metadata['brain_region'] == 'substantia_nigra')

In [ ]:
## check mixed pooled samples
f_metadata = metadata.filter(metadata["brain_region"] == "cyngulate_cortex.substantia_nigra")
mix = []
for row in f_metadata.iter_rows(named=True):  # named=True gives a dictionary
    numbers = row["pool"].split("_")  # Replace 'pool' with the actual column name
    for n in numbers:
        mix.append(n)  # Process as needed
mix_a = [x for x in mix if 'a' in x]
mix_b = [x for x in mix if 'b' in x]

In [35]:
#metadata_add = metadata.filter(metadata['brain_region'] == 'substantia_nigra')[['pool','technology']]
# Filter rows where 'brain_region' is either 'substantia_nigra' or 'cyngulate_cortex.substantia_nigra'
metadata_add = metadata.filter(
    (metadata["brain_region"] == "substantia_nigra") | 
    (metadata["brain_region"] == "cyngulate_cortex.substantia_nigra")
).select(["pool", "technology"])  # Select only 'pool' and 'technology' columns
metadata_add = pd.DataFrame(metadata_add)
metadata_add = metadata_add.rename(columns={
    0: 'pool__',
    1: 'technology'
})

In [39]:
technique_map = {'MultiomeATAC':'tech_MultiomeATAC',
'scATAC':'tech_scATAC',
'ScaleBioIH':'tech_Scaled',
'HYAv2':'tech_scATAC',
'ScaleBioIH-HYAv2':'tech_Scaled',
'ScaleBioIH6':'tech_Scaled',
'ScaleBio':'tech_Scaled',}
global_tech = []
for val in metadata_add['technology']:
    for key in technique_map.keys():
        if val == key:
            global_tech.append(technique_map[key])
metadata_add['global_tech'] = global_tech   

In [ ]:
import pandas as pd
from collections import defaultdict

pool_dict = defaultdict(list)

for _, row in metadata_add.iterrows():
    numbers = row["pool__"].split("_")  # Split numbers
    for number in numbers:
        if row["global_tech"] not in pool_dict[number]:  # Avoid duplicates
            pool_dict[number].append(row["global_tech"])

result_dict_ = dict(pool_dict)

print(len(result_dict_.keys()))
result_dict = {}
for k in result_dict_.keys():
    if 'a' in k:
      result_dict[k] = result_dict_[k]

In [ ]:
##sanity check for donors in mixed pools 
for k in mix:
    if 'a' in k: 
        print(k)
        if k in result_dict.keys():
            print('found')
        else:
            print('not found')
        print()

In [ ]:
new_dict = {}
for key, value in result_dict.items():
    number = int(key[:-1])  # Remove 'b' and convert to int
    if number < 10:
        new_key = f"ASA_00{number}"
    elif number < 100:
        new_key = f"ASA_0{number}"
    else:
        new_key = f"ASA_{number}"
    new_dict[new_key] = value

In [45]:
df_metadata = donor_info

# Create an empty DataFrame for methods
df_methods = pd.DataFrame(0, index=["tech_scATAC", "tech_Scaled", "tech_MultiomeATAC"], columns=df_metadata.columns)

# Populate the method rows based on the new_dict
for asa_key in df_methods.columns:
    if asa_key in new_dict.keys():
        for tech in new_dict[asa_key]:  # Get the associated technologies
            if tech in df_methods.index:
                df_methods.loc[tech, asa_key] = 1

# Combine metadata and method assignments into the final DataFrame
final_df = pd.concat([df_metadata, df_methods])

In [48]:
## split per cell type
covariates_per_celltype = {}
for ct in loaded_ct_region_donor_per_ct_matrices.keys():
    donors = [x for x in loaded_ct_region_donor_per_ct_matrices[ct]['donors_filtered'] if 'ASA' in x]
    covariates_per_celltype[ct] = pd.DataFrame(index = final_df.index, columns = donors)
    for d in donors:
        for c in final_df.columns:
            if d == c:
                covariates_per_celltype[ct][d] = final_df[c]

for ct in covariates_per_celltype.keys():
    output_filename = proj_subdir + f"data/covariates/{ct}_cov.txt"
    covariates_per_celltype[ct].to_csv(save_dir + output_filename, sep='\t', header=True, index=True)

### Add genotype PC

In [ ]:
snps_donor_pca = data_dir + 'plink/WGS_chm13.snps_only_genotype_pca.eigenvec'
snps_donor_pca_df = pd.read_csv(snps_donor_pca, delim_whitespace=True)
snps_donor_pca_df.index = snps_donor_pca_df['#IID']
snps_donor_pca_df = snps_donor_pca_df.drop(columns=['#IID'])
snps_donor_pca_df = snps_donor_pca_df.T
snps_donor_pca_df.index = [f'SNP_{x}' for x in snps_donor_pca_df.index]

In [87]:
covariates_per_celltype_SNP = {}

for ct in covariates_per_celltype.keys():
    df = covariates_per_celltype[ct].copy()
    donors = df.columns
    df_snp = snps_donor_pca_df[donors]
    covariates_per_celltype_SNP[ct] = pd.concat([df, df_snp], ignore_index=False)

In [95]:
for ct in covariates_per_celltype_disease.keys():
    covariates_per_celltype_SNP[ct].to_csv(proj_subdir + f'data/covariates/{ct}_cov_SNP.txt', sep='\t', header=True, index=True)

### Add data PCA

In [96]:
from sklearn.decomposition import PCA
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
## take the first 30 PCs
data_pca = {}
for ct in loaded_ct_region_donor_per_ct_matrices.keys():
    print(ct)
    data_pca[ct] = {}
    
    transposed_matrix = loaded_ct_region_donor_per_ct_matrices[ct]['matrix'].T  # Now rows are samples and columns are regions

    centered_matrix = transposed_matrix - np.mean(transposed_matrix, axis=1, keepdims=True)
    
    # Check the first few rows of the centered data
    scaler = StandardScaler()
    scaled_matrix = scaler.fit_transform(centered_matrix)
    if len(loaded_ct_region_donor_per_ct_matrices[ct]['donors_filtered']) > 30:
        pca = PCA(n_components=30)
        pca_results = pca.fit_transform(scaled_matrix)
        
        # 3. Create a DataFrame for the PCA results
        pca_df = pd.DataFrame(data=pca_results, index=loaded_ct_region_donor_per_ct_matrices[ct]['donors_filtered'], columns=[f'PC{i+1}' for i in range(pca_results.shape[1])])
        
        # 4. Print the explained variance ratio for each component
        explained_variance_ratio = pca.explained_variance_ratio_
        #print("Explained Variance Ratio per component:", explained_variance_ratio)
        total_variance_explained = np.sum(explained_variance_ratio[:30])
        print(f"Total variance explained by the first 30 PCs for {ct}: {total_variance_explained * 100:.2f}%")
    
        data_pca[ct]['30PCs_var'] = total_variance_explained
        data_pca[ct]['PCs'] = pca_df

In [101]:
explained_variance = []
explained_variance_cts = []
for k in data_pca.keys():
    ##only keep cell types with over 30 donors
    if len(loaded_ct_region_donor_per_ct_matrices[k]['donors_filtered']) > 30:
        explained_variance.append(data_pca[k]['30PCs_var'])
        explained_variance_cts.append(k)

In [ ]:
for k in data_pca.keys():
    if len(loaded_ct_region_donor_per_ct_matrices[k]['donors_filtered']) > 30:
        df = data_pca[k]['PCs'].T
        df.index = [f'Acc_{x}' for x in df.index]
        data_pca[k]['PCs_form'] = df

In [ ]:
covariates_per_celltype_SNP

In [ ]:
covariates_per_celltype_SNP_dataPCA = {}
for ct in data_pca.keys():
    if len(loaded_ct_region_donor_per_ct_matrices[ct]['donors_filtered']) > 30:
        print(ct)
        df = data_pca[ct]['PCs_form']
        print(f'OG shape: {covariates_per_celltype_SNP[ct].shape}')   
        print(f'data pca shape: {df.shape}')
        df_filtered = df[covariates_per_celltype_SNP[ct].columns]
        print(f'data pca filtered shape: {df.shape}') 
        donor_overlap = len(list(set(covariates_per_celltype_SNP[ct].columns)&set(df_filtered.columns)))
        print(f'Donor overlap: {donor_overlap}/{covariates_per_celltype_SNP[ct].shape[1]}')
        if covariates_per_celltype_SNP[ct].columns.equals(df_filtered.columns):
            print('Columns order is the same')
        else:
            print('####################################')

        covariates_per_celltype_SNP_dataPCA[ct] = pd.concat([covariates_per_celltype_SNP[ct], df_filtered], ignore_index=False)
        print(f'OG shape: {covariates_per_celltype_SNP[ct].shape}')  
        final_shape = covariates_per_celltype_SNP_dataPCA[ct].shape
        print(f'final shape: {final_shape}')   
        print()

In [118]:
for ct in data_pca.keys():
    if len(loaded_ct_region_donor_per_ct_matrices[ct]['donors_filtered']) > 30:
        covariates_per_celltype_SNP_dataPCA[ct].to_csv(proj_subdir + f'data/covariates/{ct}_cov_SNP_data_pca_30PCs.txt', sep='\t', header=True, index=True)

## 3. run tensorQTL

In [ ]:
! tensorQTL/tensorQTL_parquet.sh SNP genotype_data_pca CT2 SN